In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.plotting import add_at_risk_counts


def plot_km_from_result_dict(
    data,
    risk_higher_means_worse=True,
    title="Kaplan-Meier Survival Curve",
    save_path=None
):

    rows = []
    for pid, info in data.items():
        rows.append({
            "patient_id": pid,
            "risk": float(info["risk"]),
            "survival": float(info["survival"]),
            "censorship": float(info["censorship"]),
        })

    df = pd.DataFrame(rows)

    df["event"] = 1 - df["censorship"]

    risk_median = df["risk"].median()

    if risk_higher_means_worse:
        df["risk_group"] = np.where(df["risk"] >= risk_median, "High risk", "Low risk")
    else:
        df["risk_group"] = np.where(df["risk"] < risk_median, "High risk", "Low risk")

    high = df[df["risk_group"] == "High risk"].copy()
    low = df[df["risk_group"] == "Low risk"].copy()

    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()

    plt.figure(figsize=(6, 6))

    kmf_high.fit(
        durations=high["survival"],
        event_observed=high["event"],
        label=f"High"
    )
    ax = kmf_high.plot_survival_function(ci_show=True)

    kmf_low.fit(
        durations=low["survival"],
        event_observed=low["event"],
        label=f"Low"
    )
    kmf_low.plot_survival_function(ci_show=True, ax=ax)

    add_at_risk_counts(kmf_high, kmf_low, ax=ax, rows_to_show=["At risk"])

    plt.subplots_adjust(bottom=0.15)

    result = logrank_test(
        high["survival"],
        low["survival"],
        event_observed_A=high["event"],
        event_observed_B=low["event"]
    )

    plt.title(title)
    # plt.xlabel("Time (months)")
    plt.ylabel("Survival probability")
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.25)
    plt.legend(frameon=False)

    text_str = (
        f"Risk median = {risk_median:.4f}\n"
        f"log-rank p = {result.p_value:.4g}\n"
    )
    plt.text(
        0.04, 0.04,
        text_str,
        transform=plt.gca().transAxes,
        fontsize=10,
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    print("========== KM summary ==========")
    print(f"Total patients: {len(df)}")
    print(f"Risk median: {risk_median:.6f}")
    print(f"High risk n = {len(high)}, events = {int(high['event'].sum())}, censored = {int(high['censorship'].sum())}")
    print(f"Low  risk n = {len(low)}, events = {int(low['event'].sum())}, censored = {int(low['censorship'].sum())}")
    print(f"log-rank p-value: {result.p_value:.6g}")
    print(f"High risk median survival: {kmf_high.median_survival_time_}")
    print(f"Low  risk median survival: {kmf_low.median_survival_time_}")

    return {
        "df": df,
        "risk_median": risk_median,
        "logrank_p": result.p_value,
        "kmf_high": kmf_high,
        "kmf_low": kmf_low,
    }

def save_risk_groups_to_csv(data, save_path, risk_higher_means_worse=True):

    rows = []
    for pid, info in data.items():
        rows.append({
            "patient_id": pid,
            "risk": float(info["risk"]),
            "survival_time": float(info["survival"]),
            "censorship": int(info["censorship"]),  # 1=删失, 0=死亡
        })

    df = pd.DataFrame(rows)

    df["event"] = 1 - df["censorship"]

    risk_median = df["risk"].median()

    if risk_higher_means_worse:
        df["risk_group"] = np.where(df["risk"] >= risk_median, "High", "Low")
    else:
        df["risk_group"] = np.where(df["risk"] < risk_median, "High", "Low")

    # 保存
    df.to_csv(save_path, index=False)

    print(f"[Saved CSV] {save_path}")
    print(f"Risk median: {risk_median:.6f}")
    print(df.head())

    return df

In [ ]:
import pickle
import numpy as np

#CombinedModel
# train_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/CombinedModel_nll_surv_a0.0_5foldcv_gc32/lihc_CombinedModel_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_train_2_results.pkl"
# val_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/CombinedModel_nll_surv_a0.0_5foldcv_gc32/lihc_CombinedModel_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_val_2_results.pkl"
# test_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/CombinedModel_nll_surv_a0.0_5foldcv_gc32/lihc_CombinedModel_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_test_2_results.pkl"

#PatchGCN
# train_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/PatchGCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_PatchGCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_train_2_results.pkl"
# val_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/PatchGCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_PatchGCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_val_2_results.pkl"
# test_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/PatchGCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_PatchGCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_test_2_results.pkl"

# H2GCN
train_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/H2GCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_H2GCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_train_2_results.pkl"
val_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/H2GCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_H2GCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_val_2_results.pkl"
test_file = "/root/Desktop/data/private/hjx_product/results_check/5foldcv/H2GCN_nll_surv_a0.0_5foldcv_gc32_spatial/lihc_H2GCN_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_test_2_results.pkl"

# AMIL
# train_file = "/root/Desktop/data/private/hjx_product/results_final_0614/5foldcv/AMIL_nll_surv_a0.0_5foldcv_gc32/lihc_AMIL_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_train_2_results.pkl"
# val_file = "/root/Desktop/data/private/hjx_product/results_final_0614/5foldcv/AMIL_nll_surv_a0.0_5foldcv_gc32/lihc_AMIL_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_val_2_results.pkl"
# test_file = "/root/Desktop/data/private/hjx_product/results_final_0614/5foldcv/AMIL_nll_surv_a0.0_5foldcv_gc32/lihc_AMIL_nll_surv_a0.0_5foldcv_gc32_s42/split_latest_test_2_results.pkl"


with open(train_file, "rb") as f:
    train_data = pickle.load(f)

with open(val_file, "rb") as f:
    val_data = pickle.load(f)

with open(test_file, "rb") as f:
    test_data = pickle.load(f)


combined_data = {}
combined_data.update(train_data)
combined_data.update(val_data)
combined_data.update(test_data)

In [ ]:
result = plot_km_from_result_dict(
    combined_data,
    risk_higher_means_worse=True,
    title="H2GCN KM Curve",
    save_path="/root/Desktop/data/private/hjx_product/results_final_kmcurves/H2GCN_km_curve.png"
)

csv_path = "/root/Desktop/data/private/hjx_product/results_final_kmcurves/H2GCN_risk_groups.csv"

df = save_risk_groups_to_csv(
    combined_data,
    save_path=csv_path,
    risk_higher_means_worse=True
)
df.groupby("risk_group")[["event", "survival_time"]].mean()